# 09 RAG Eval Case Builder

Extract the two annual reports with Azure Document Intelligence, save the raw markdown, build 10 broad coverage windows, and review the curated 10-question financial eval set before wiring the questions into vector DB retrieval evaluation.

## Scope

- Target reports: `bandhan_annual_report.pdf` and `emcure_annual_report.pdf`
- Output markdown: `artifacts/rag_eval/raw_markdown/`
- Output eval set: `data/evals/fundamental_rag_eval_cases.json`
- This notebook does not score retrieval yet; it prepares the grounded question-answer cases.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import Markdown, display
import pandas as pd

from market_analyst.services.rag_eval import (
    build_broad_eval_chunks,
    build_eval_markdown_reports,
    eval_cases_as_rows,
    eval_chunks_as_rows,
    export_markdown_reports,
    load_rag_eval_cases,
)


## Run Configuration

Set `DOCUMENT_INTELLIGENCE_CONNECTION_VERIFY = 'false'` only when this machine cannot validate the Azure endpoint certificate chain. If the environment already trusts the certificate chain, leave it empty and the provider will use normal TLS verification.

In [ ]:
DOCUMENT_INTELLIGENCE_CONNECTION_VERIFY = os.getenv('DOCUMENT_INTELLIGENCE_CONNECTION_VERIFY', '')
MAX_PAGES = None
TARGET_CHUNKS_PER_REPORT = 5
MARKDOWN_OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'rag_eval' / 'raw_markdown'

os.environ['DOCUMENT_INTELLIGENCE_CONNECTION_VERIFY'] = DOCUMENT_INTELLIGENCE_CONNECTION_VERIFY
MARKDOWN_OUTPUT_DIR

In [ ]:
markdown_reports = build_eval_markdown_reports(PROJECT_ROOT / 'reports', max_pages=MAX_PAGES)
written_markdown_paths = export_markdown_reports(markdown_reports, MARKDOWN_OUTPUT_DIR)

markdown_summary = pd.DataFrame(
    [
        {
            'company_name': report.report.company_name,
            'report_file': report.report.path.name,
            'page_count': report.page_count,
            'markdown_characters': len(report.markdown),
            'markdown_path': str(path.relative_to(PROJECT_ROOT)),
        }
        for report, path in zip(markdown_reports, written_markdown_paths, strict=True)
    ]
)
markdown_summary

## Raw Markdown Preview

In [ ]:
for report in markdown_reports:
    print(report.report.path.name)
    display(Markdown(report.markdown[:4000]))
    print('-' * 80)


## Broad Coverage Windows

These are intentionally larger windows built from consecutive markdown pages so the next eval step can check whether retrieval covers broad report regions rather than only tiny snippets.

In [ ]:
broad_chunks = build_broad_eval_chunks(markdown_reports, target_chunks_per_report=TARGET_CHUNKS_PER_REPORT)
broad_chunks_df = pd.DataFrame(eval_chunks_as_rows(broad_chunks))
broad_chunks_df[['company_name', 'report_file', 'chunk_index', 'start_page', 'end_page', 'page_count', 'character_count']]

In [ ]:
for company_name in broad_chunks_df['company_name'].unique():
    chunk = next(item for item in broad_chunks if item.company_name == company_name)
    print(company_name, f'pages {chunk.start_page}-{chunk.end_page}')
    display(Markdown(chunk.content[:3500]))
    print('-' * 80)


## Curated Financial Eval Cases

In [ ]:
eval_cases = load_rag_eval_cases(PROJECT_ROOT / 'data' / 'evals' / 'fundamental_rag_eval_cases.json')
eval_cases_df = pd.DataFrame(eval_cases_as_rows(eval_cases))
eval_cases_df[['case_id', 'company_name', 'question_style', 'evaluation_focus', 'question', 'expected_answer', 'source_pages']]

In [ ]:
eval_cases_df.groupby(['company_name', 'question_style']).size().rename('case_count').reset_index()

## Validation

In [ ]:
assert len(markdown_reports) == 2
assert all(path.exists() for path in written_markdown_paths)
assert len(broad_chunks) == 10
assert len(eval_cases) == 10
assert set(eval_cases_df['company_name']) == {'Bandhan', 'Emcure'}
print('RAG eval notebook prepared markdown outputs, broad chunks, and 10 curated eval cases.')